In [ ]:
# # ! Setup
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyshtools as pysh
import pygmt
import pyvista as pv
import time
from gravity_forward_numba import WerSch_numba

In [ ]:
# # ! Constants
myG = 6.67430e-11
G = pysh.constants.G.value
gm_moon = pysh.constants.Moon.gm.value
r_calc = 1748000.0

In [ ]:
# # ! Shape of Moon
#### * Shape
lmax_shp = 719
clm_shp_moon = pysh.SHCoeffs.from_file(f'Moon_shape_{lmax_shp}.sh', 
                                       lmax=lmax_shp, 
                                       name='LOLA_shape (Moon)',
                                       units='m', format='bshc')
r_itfc = clm_shp_moon.coeffs[0,0,0]

In [ ]:
# # ! Computation of topographic potential: Spectral-domain 
lmax_shp = 360
rho0 = 2560.0 # kg/m^3
nmaxs = np.arange(1, 7)
clms = []
for nmax in nmaxs:
    t0 = time.time()
    lmax = (lmax_shp+1)*nmax-1
    clm_topograv_moon = \
        pysh.SHGravCoeffs.from_shape(shape=clm_shp_moon,
                                     rho=rho0,
                                     gm=gm_moon,
                                     nmax=nmax,
                                     lmax=lmax,
                                     lmax_grid=lmax,
                                     lmax_calc=lmax_shp,
                                     name=f'LOLA_Topo_Grav (Moon) nmax={nmax}')
    tc = time.time() - t0
    clms.append(clm_topograv_moon)
    print(f"nmax = {nmax:2d}; lmax = {lmax:2d}; time cost: {tc:8.3f}")

nmax =  1; lmax = 360; time cost:    0.085
nmax =  2; lmax = 721; time cost:    0.848
nmax =  3; lmax = 1082; time cost:    4.830
nmax =  4; lmax = 1443; time cost:   16.685
nmax =  5; lmax = 1804; time cost:   43.627
nmax =  6; lmax = 2165; time cost:   90.939


In [ ]:
# # ! Setup
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyshtools as pysh
import pygmt
import pyvista as pv
import time
from gravity_forward_numba import WerSch_numba

In [ ]:
# # ! Constants
myG = 6.67430e-11
G = pysh.constants.G.value
gm_moon = pysh.constants.Moon.gm.value
r_calc = 1748000.0

In [ ]:
# # ! Shape of Moon
#### * Shape
lmax_shp = 719
clm_shp_moon = pysh.SHCoeffs.from_file(f'Moon_shape_{lmax_shp}.sh', 
                                       lmax=lmax_shp, 
                                       name='LOLA_shape (Moon)',
                                       units='m', format='bshc')
r_itfc = clm_shp_moon.coeffs[0,0,0]

In [ ]:
# # ! Computation of topographic potential: Spectral-domain 
rho0 = 2560.0 # kg/m^3
nmaxs = np.arange(1, 7)
clms = []
for nmax in nmaxs:
    t0 = time.time()
    lmax = (lmax_shp+1)*nmax-1
    clm_topograv_moon = \
        pysh.SHGravCoeffs.from_shape(shape=clm_shp_moon,
                                     rho=rho0,
                                     gm=gm_moon,
                                     nmax=nmax,
                                     lmax=lmax,
                                     lmax_grid=lmax,
                                     lmax_calc=lmax_shp,
                                     name=f'LOLA_Topo_Grav (Moon) nmax={nmax}')
    tc = time.time() - t0
    clms.append(clm_topograv_moon)
    print(f"nmax = {nmax:2d}; lmax = {lmax:2d}; time cost: {tc:8.3f}")

nmax =  1; lmax = 719; time cost:    0.425
nmax =  2; lmax = 1439; time cost:    9.277
nmax =  3; lmax = 2159; time cost:   48.896
nmax =  4; lmax = 2879; time cost:  141.456
nmax =  5; lmax = 3599; time cost:  329.001
nmax =  6; lmax = 4319; time cost:  674.230


In [ ]:
# # ! Single contribution of each nmax in a band range
print("nmax | l-beg | l-end | t-expand |  min_gz  |  max_gz  |  mean_gz |  std_gz")
for i_clm in np.arange(len(clms)):
    nmax = nmaxs[i_clm]
    if i_clm == 0:
        dif_clm = clms[i_clm]
    else:
        lmax = (lmax_shp+1)*(nmax-1)-1
        clm_1 = clms[i_clm-1]
        clm_2 = clms[i_clm]
        dif_clm = clm_2.copy()
        dif_clm.coeffs[:,:lmax,:lmax] = \
            clm_2.coeffs[:,:lmax,:lmax] - clm_1.coeffs[:,:lmax,:lmax]
    for n_cut in np.arange(nmax):
        lmax1 = (lmax_shp+1)*n_cut-1
        lmax2 = (lmax_shp+1)*(n_cut+1)-1
        dif_clm.coeffs[:, :lmax1+1, :lmax1+1] = 0.0
        dif_clm.coeffs[:, lmax2+1:, lmax2+1:] = 0.0
        t0 = time.time()
        grd_sc = dif_clm.expand(a=r_calc, 
                                f=0.0,
                                lmax_calc=lmax2)
        t1 = time.time() - t0
        gz = -1.e5 * grd_sc.rad
        min_gz  = np.min(gz.data)
        max_gz  = np.max(gz.data)
        mean_gz = np.mean(gz.data)
        std_gz  = np.std(gz.data)
        fmt = ("{:4d} | {:5d} | {:5d} | {:8.4f} | "
               "{:8.3f} | {:8.3f} | {:8.3f} | {:8.3f}")
        print(fmt.format(nmax, lmax1+1, lmax2, t1,
                         min_gz, max_gz, mean_gz, std_gz))
#### * Mapping
# fig = pygmt.Figure()
# gz_topo.plotgmt(fig=fig,
#                 projection='mollweide',
#                 central_longitude=-90.,
#                 grid=[30, 30],
#                 tick_interval=None,
#                 cmap='vik',
#                 cmap_limits=[-800, 800],
#                 colorbar='bottom',
#                 cb_triangles='both',
#                 cb_label='Topographic radial gravity (mGal)',
#                 cb_tick_interval=200,
#                 cb_minor_tick_interval=100)
# fig.show(width=800)

nmax | l-beg | l-end | t-expand |  min_gz  |  max_gz  |  mean_gz |  std_gz
   1 |     0 |   719 |   2.1557 | -768.370 | 1029.410 |  -35.580 |  245.966
   2 |     0 |   719 |   4.5533 |  -50.168 |  108.739 |    0.544 |    6.858
   2 |   720 |  1439 |  16.1788 |   -0.506 |    0.554 |    0.000 |    0.037
   3 |     0 |   719 |   7.6969 |  -29.558 |   46.067 |    0.001 |    1.280
   3 |   720 |  1439 |  25.1122 |   -1.206 |    1.004 |    0.000 |    0.036
   3 |  1440 |  2159 |  53.3826 |   -0.002 |    0.001 |    0.000 |    0.000
   4 |     0 |   719 |  13.1086 |  -17.835 |   24.488 |   -0.000 |    0.370
   4 |   720 |  1439 |  35.2318 |   -2.124 |    1.708 |   -0.000 |    0.034
   4 |  1440 |  2159 |  72.0521 |   -0.004 |    0.004 |   -0.000 |    0.000
   4 |  2160 |  2879 | 124.8830 |      nan |      nan |      nan |      nan
   5 |     0 |   719 |  17.8823 |  -10.614 |   12.922 |   -0.000 |    0.136
   5 |   720 |  1439 |  46.8995 |   -2.519 |    2.284 |    0.000 |    0.027
   5 |  1440 